In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
SEED = 42
np.random.seed(SEED)

# 1. Load the raw dataset
df = pd.read_csv(
    r'C:\Users\muthu\OneDrive\Desktop\ML-Capstone-Project\data\raw\data.csv',
    encoding='ISO-8859-1'
)

print("=" * 60)
print("SECTION A - DATASET AUDIT")
print("=" * 60)
print(f"Original Shape: {df.shape}")
print("\nData Types & Missing Values:")
print(df.info())
print("\nMissing Value Counts:\n", df.isnull().sum())
print("\nSummary Statistics:\n", df.describe().T)


# 2. Filter out cancellations, returns, and invalid pricing anomalies
df_clean = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].copy()

# 3. Handle date formatting
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], errors='coerce')

In [ ]:
# ============================================================
# MANDATORY EDA VISUALIZATIONS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Target Distribution Plot (UnitPrice)
sns.histplot(df_clean['UnitPrice'], kde=True, ax=axes[0, 0], color='teal', log_scale=True)
axes[0, 0].set_title("Distribution of Unit Price (Log Scale)")
axes[0, 0].set_xlabel("Unit Price ($)")
axes[0, 0].set_ylabel("Frequency")

# 2. Correlation Heatmap for Numerical Columns
numeric_eda_cols = df_clean.select_dtypes(include=[np.number]).columns
sns.heatmap(df_clean[numeric_eda_cols].corr(), annot=True, cmap='mako', fmt=".2f", ax=axes[0, 1])
axes[0, 1].set_title("Feature Correlation Heatmap")

# 3. Scatter Plot 1: Quantity vs UnitPrice
sns.scatterplot(
    data=df_clean.sample(min(10000, len(df_clean)), random_state=SEED), 
    x='Quantity', 
    y='UnitPrice', 
    ax=axes[1, 0], 
    color='indigo', 
    alpha=0.4
)
axes[1, 0].set_title("Quantity vs. Unit Price (Sampled)")
axes[1, 0].set_xlabel("Quantity")
axes[1, 0].set_ylabel("Unit Price ($)")
axes[1, 0].set_yscale('log')

# 4. Boxplot of UnitPrice by Top Countries
top_countries = df_clean['Country'].value_counts().head(6).index
sns.boxplot(
    data=df_clean[df_clean['Country'].isin(top_countries)],
    x='Country',
    y='UnitPrice',
    hue='Country',
    legend=False,
    ax=axes[1, 1],
    palette='Set2'
)
axes[1, 1].set_title("Unit Price Distribution Across Top Countries")
axes[1, 1].set_xlabel("Country")
axes[1, 1].set_ylabel("Unit Price ($)")
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer


# ============================================================
# SECTION B : DATA PREPROCESSING
# ============================================================

SEED = 42
TARGET = 'UnitPrice'


# ============================================================
# B1. DATA LOADING
# ============================================================

df = pd.read_csv(
    r'C:\Users\muthu\OneDrive\Desktop\Amrita\Third year\ML\data_3.csv',
    encoding='ISO-8859-1'
)

print("=" * 60)
print("INITIAL DATASET")
print("=" * 60)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# B1. MISSING VALUE ANALYSIS
# ============================================================

print("\n" + "=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)

print(df.isnull().sum())


# Handle missing categorical values
if 'Description' in df.columns:
    df['Description'] = df['Description'].fillna('UNKNOWN_PRODUCT')

# CustomerID is not used later, so no need to impute it.
# It will be removed as an identifier.


# ============================================================
# B1. DUPLICATE HANDLING
# ============================================================

duplicate_count = df.duplicated().sum()

print("\nDuplicate rows found:", duplicate_count)

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after duplicate removal:", df.shape)


# ============================================================
# B1. INVALID VALUE / PHYSICAL ANOMALY HANDLING
# ============================================================

# Quantity and UnitPrice should be positive for this prediction task.

initial_rows = len(df)

df = df[
    (df['Quantity'] > 0) &
    (df['UnitPrice'] > 0)
].reset_index(drop=True)

removed_rows = initial_rows - len(df)

print("\nInvalid records removed:", removed_rows)
print("Shape after validity filtering:", df.shape)


# ============================================================
# B1. DATE VALIDATION
# ============================================================

df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate'],
    errors='coerce'
)

invalid_dates = df['InvoiceDate'].isna().sum()

print("\nInvalid InvoiceDate values:", invalid_dates)

# Remove records with invalid dates
df = df.dropna(
    subset=['InvoiceDate']
).reset_index(drop=True)

print("Shape after date cleaning:", df.shape)


# ============================================================
# B1. OUTLIER ANALYSIS
# ============================================================

print("\n" + "=" * 60)
print("OUTLIER ANALYSIS")
print("=" * 60)

Q1 = df[TARGET].quantile(0.25)
Q3 = df[TARGET].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outlier_mask = (
    (df[TARGET] < lower_bound) |
    (df[TARGET] > upper_bound)
)

outlier_count = outlier_mask.sum()

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of UnitPrice outliers:", outlier_count)

# We do NOT blindly delete legitimate high-price observations.
# Instead, the target is log-transformed below to reduce
# the influence of extreme right-skewed values.


# ============================================================
# B3. FEATURE ENGINEERING
# ============================================================

print("\n" + "=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

# Extract useful temporal information from InvoiceDate

df['year'] = df['InvoiceDate'].dt.year
df['month'] = df['InvoiceDate'].dt.month
df['day_of_week'] = df['InvoiceDate'].dt.dayofweek
df['hour'] = df['InvoiceDate'].dt.hour

print("Created features:")
print("- year")
print("- month")
print("- day_of_week")
print("- hour")


# ============================================================
# REMOVE IDENTIFIER / HIGH-CARDINALITY COLUMNS
# ============================================================

df.drop(
    columns=[
        'InvoiceDate',
        'InvoiceNo',
        'StockCode',
        'Description',
        'CustomerID'
    ],
    inplace=True,
    errors='ignore'
)


# ============================================================
# B2. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop(columns=[TARGET])

# Preserve original target for final evaluation
y_original = df[TARGET].copy()

# Log transformation to reduce target skewness
y = np.log1p(y_original)


# ============================================================
# B2. STRATIFIED TRAIN / TEST SPLIT
# ============================================================

# Regression targets are continuous, so create quantile bins
# for stratification.

y_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates='drop'
)

(
    X_train,
    X_test,
    y_train,
    y_test,
    y_train_original,
    y_test_original
) = train_test_split(
    X,
    y,
    y_original,
    test_size=0.20,
    random_state=SEED,
    stratify=y_bins
)

print("\n" + "=" * 60)
print("TRAIN / TEST SPLIT")
print("=" * 60)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


# ============================================================
# B2. RARE CATEGORY HANDLING
# ============================================================

if 'Country' in X_train.columns:

    # Calculate category frequency ONLY from training data
    country_counts = X_train['Country'].value_counts()

    # Countries with fewer than 5 training observations
    # are grouped into "Other"
    rare_countries = country_counts[
        country_counts < 5
    ].index

    X_train['Country'] = X_train['Country'].replace(
        rare_countries,
        'Other'
    )

    X_test['Country'] = X_test['Country'].replace(
        rare_countries,
        'Other'
    )

    print("\nRare countries grouped:", len(rare_countries))


# ============================================================
# B2. IDENTIFY FEATURE TYPES
# ============================================================

categorical_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

print("\nNumerical features:")
print(numerical_cols)

print("\nCategorical features:")
print(categorical_cols)


# ============================================================
# B2. SCALING + ENCODING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            StandardScaler(),
            numerical_cols
        ),

        (
            'cat',
            OneHotEncoder(
                drop='first',
                handle_unknown='ignore',
                sparse_output=False
            ),
            categorical_cols
        )
    ]
)


# ============================================================
# B2. FIT ONLY ON TRAINING DATA
# ============================================================

# IMPORTANT:
# The scaler and encoder learn parameters only from X_train.

X_train_processed = preprocessor.fit_transform(X_train)

# Test data is transformed using the already-fitted
# preprocessing objects.

X_test_processed = preprocessor.transform(X_test)


# ============================================================
# B2. CREATE FINAL PROCESSED DATAFRAMES
# ============================================================

feature_names = preprocessor.get_feature_names_out()

X_train_scaled = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)


# ============================================================
# FINAL VALIDATION
# ============================================================

print("\n" + "=" * 60)
print("FINAL PREPROCESSING VALIDATION")
print("=" * 60)

print("Original dataset shape:", df.shape)

print("Processed X_train shape:", X_train_scaled.shape)
print("Processed X_test shape :", X_test_scaled.shape)

print("\nMissing values in X_train:")
print(X_train_scaled.isnull().sum().sum())

print("\nMissing values in X_test:")
print(X_test_scaled.isnull().sum().sum())

print("\nTarget transformation:")
print("Original target mean:", y_original.mean())
print("Original target median:", y_original.median())
print("Log-transformed target mean:", y.mean())

print("\nSection B preprocessing completed successfully.")

print("Processed Features Head:")
print(X_train_scaled.head())